# TPOT Mercedes Prediction
Predicting the time a Mercedes-Benz car spends on the test bench.

## Project Overview
Regression project using the Mercedes-Benz Greener Manufacturing dataset to predict testing time.

## Learning Objectives
- Handle high-dimensional sparse binary features
- Reduce dimensionality for regression
- Compare boosting models on manufacturing data

## Problem Statement
Predict the time (seconds) each car configuration spends on the test bench, reducing total testing time.

## Why This Project Matters
Faster testing means lower CO2 emissions and energy use in manufacturing. Optimizing test bench time is a green manufacturing goal.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | Local: mercedesbenz.csv |
| **Target** | y (test bench time in seconds) |
| **Type** | Regression |
| **Features** | ~370 mixed categorical + binary features |

## Dataset Source & License
Mercedes-Benz Greener Manufacturing Kaggle competition dataset.

## Environment Setup

In [1]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Imports

In [2]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
from sklearn.decomposition import PCA
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [3]:
TARGET = 'y'

## Dataset Loading

In [4]:
csv_path = os.path.join(SAVE_DIR, 'mercedesbenz.csv')
if not os.path.exists(csv_path):
    raise FileNotFoundError(f'Dataset not found at {csv_path}')
df = pd.read_csv(csv_path)
print(f'Shape: {df.shape}')
df.head()

Shape: (4209, 378)


,ID,y,X0,X1,X2,X3,X4,X5,X6,X8,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,0,130.81,k,v,at,a,d,u,j,o,...,0,0,1,0,0,0,0,0,0,0
1,6,88.53,k,t,av,e,d,y,l,o,...,1,0,0,0,0,0,0,0,0,0
2,7,76.26,az,w,n,c,d,x,j,x,...,0,0,0,0,0,0,1,0,0,0
3,9,80.62,az,t,n,f,d,x,l,e,...,0,0,0,0,0,0,0,0,0,0
4,13,78.02,az,v,n,f,d,h,d,n,...,0,0,0,0,0,0,0,0,0,0


## Data Validation

In [5]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')
print(f'\nTarget stats:')
print(df[TARGET].describe())

Missing: 0
Duplicates: 0

Target stats:
count    4209.000000
mean      100.669318
std        12.679381
min        72.110000
25%        90.820000
50%        99.150000
75%       109.010000
max       265.320000
Name: y, dtype: float64


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df[TARGET].hist(bins=50, ax=axes[0], edgecolor='black')
axes[0].set_title('Target (y) Distribution')

# Categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns: {cat_cols}')
if cat_cols:
    df[cat_cols[0]].value_counts().head(15).plot.bar(ax=axes[1])
    axes[1].set_title(f'{cat_cols[0]} Distribution')

# Binary feature sum
bin_cols = [c for c in df.columns if df[c].nunique() == 2 and c != TARGET]
axes[2].hist(df[bin_cols].sum(axis=1), bins=30, edgecolor='black')
axes[2].set_title('Sum of Binary Features per Row')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

Categorical columns: ['X0', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X8']


## Target Analysis

In [7]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')
print(f'Outliers (>200s): {(df[TARGET]>200).sum()}')

count    4209.000000
mean      100.669318
std        12.679381
min        72.110000
25%        90.820000
50%        99.150000
75%       109.010000
max       265.320000
Name: y, dtype: float64

Skewness: 1.21
Outliers (>200s): 1


## Train/Test Split

In [8]:
# Drop ID
if 'ID' in df.columns:
    df = df.drop(columns=['ID'])

# Encode categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    df[c] = df[c].astype('category').cat.codes

# Remove zero-variance columns
var_zero = [c for c in df.columns if c != TARGET and df[c].nunique() <= 1]
df = df.drop(columns=var_zero)
print(f'Dropped {len(var_zero)} zero-variance columns')

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Dropped 12 zero-variance columns
Train: (3367, 364), Test: (842, 364)


## Preprocessing
Dropped ID, encoded categoricals, removed zero-variance binary columns.

## Feature Engineering
High-dimensional binary features used directly — tree models handle sparse binaries well.

## Baseline Model

In [9]:
bl = Ridge(alpha=10.0)
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline Ridge: R2={r2_score(y_test, bl_pred):.4f}  RMSE={root_mean_squared_error(y_test, bl_pred):.4f}  MAE={mean_absolute_error(y_test, bl_pred):.4f}')

Baseline Ridge: R2=0.5733  RMSE=8.1500  MAE=5.4829


## LazyPredict Benchmark

In [10]:
# --- LazyPredict Benchmark ---
try:
    from lazypredict.Supervised import LazyRegressor
    n_lp = min(5000, len(X_train))
    lr = LazyRegressor(verbose=0, ignore_warnings=True)
    lp_models, _ = lr.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

                             Adjusted R-Squared  R-Squared      RMSE  \
Model                                                                  
GradientBoostingRegressor              0.272517   0.587385  8.013961   
ElasticNetCV                           0.269962   0.585936  8.028018   
LassoCV                                0.267956   0.584798  8.039042   
OrthogonalMatchingPursuitCV            0.256341   0.578210  8.102567   
OrthogonalMatchingPursuit              0.252665   0.576125  8.122568   
BayesianRidge                          0.242621   0.570428  8.176970   
Lasso                                  0.228541   0.562442  8.252626   
PoissonRegressor                       0.227801   0.562023  8.256580   
HuberRegressor                         0.224168   0.559962  8.275984   
LarsCV                                 0.220909   0.558114  8.293346   
ElasticNet                             0.220661   0.557973  8.294668   
RidgeCV                                0.217996   0.556461  8.30

## FLAML AutoML

In [11]:
# --- FLAML AutoML ---
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='regression', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  R2={r2_score(y_test, flaml_pred):.4f}  RMSE={root_mean_squared_error(y_test, flaml_pred):.4f}  MAE={mean_absolute_error(y_test, flaml_pred):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

FLAML skipped: XGBModel.fit() got an unexpected keyword argument 'callbacks'


## Additional Models: CatBoost, LightGBM, XGBoost

In [12]:
# --- Boosting Models ---
models = {}
# CatBoost
cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

# LightGBM
lgb = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

# XGBoost
xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0)
xgb.fit(X_train, y_train)
models['XGBoost'] = xgb

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: R2={r2_score(y_test, p):.4f}  RMSE={root_mean_squared_error(y_test, p):.4f}  MAE={mean_absolute_error(y_test, p):.4f}")

CatBoost: R2=0.5854  RMSE=8.0335  MAE=5.3464
LightGBM: R2=0.5302  RMSE=8.5510  MAE=5.8013
XGBoost: R2=0.5029  RMSE=8.7966  MAE=5.8447


## Top 3 Model Selection

In [13]:
# --- Top 3 Selection ---
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {'R2': r2_score(y_test, p), 'RMSE': root_mean_squared_error(y_test, p), 'MAE': mean_absolute_error(y_test, p)}

results_df = pd.DataFrame(all_results).T.sort_values('R2', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

                R2      RMSE       MAE
CatBoost  0.585372  8.033482  5.346439
LightGBM  0.530232  8.550990  5.801251
XGBoost   0.502854  8.796631  5.844693



Top 3: ['CatBoost', 'LightGBM', 'XGBoost']


## Final Evaluation of Top 3

In [14]:
# --- Final Evaluation of Top 3 ---
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    r2 = r2_score(y_test, p)
    rmse = root_mean_squared_error(y_test, p)
    mae = mean_absolute_error(y_test, p)
    final_results[name] = {'R2': r2, 'RMSE': rmse, 'MAE': mae}
    print(f"{name}: R2={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
names = list(final_results.keys())
for i, metric in enumerate(['R2', 'RMSE', 'MAE']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

CatBoost: R2=0.5854  RMSE=8.0335  MAE=5.3464
LightGBM: R2=0.5302  RMSE=8.5510  MAE=5.8013
XGBoost: R2=0.5029  RMSE=8.7966  MAE=5.8447


## Error Analysis

In [15]:
# --- Error Analysis ---
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)
residuals = y_test - preds

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(residuals, bins=50, edgecolor='black')
axes[0].set_title('Residual Distribution')
axes[0].axvline(0, color='red', linestyle='--')
axes[1].scatter(preds, residuals, alpha=0.3, s=5)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals vs Predicted')
axes[2].scatter(y_test, preds, alpha=0.3, s=5)
mn, mx = y_test.min(), y_test.max()
axes[2].plot([mn, mx], [mn, mx], 'r--')
axes[2].set_title('Actual vs Predicted')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## Interpretation & Business Insight
Certain car configurations consistently require longer testing. Identifying these patterns helps optimize the test bench schedule.

## Limitations
- Anonymous features limit interpretability
- Small dataset (~4K rows)
- No temporal information

## How to Improve
- PCA or feature selection to reduce dimensionality
- Stacking ensemble
- Target encoding for categoricals

## Production Considerations
- Integrate into manufacturing pipeline
- Monitor prediction accuracy as new car configs appear
- Flag configs with high predicted test time

## Common Mistakes
- Not removing zero-variance features
- Overfitting on small dataset
- Ignoring categorical encoding impact

## Mini Challenge
1. Apply PCA and compare with raw features
2. Try target encoding for X0-X8
3. Remove outliers (y > 200) and retrain

## Final Summary
Predicted Mercedes test bench time from car configuration features. Boosting models handle the sparse binary feature space effectively.

In [16]:
# --- Save Metrics ---
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))

Saved metrics.json
{
  "CatBoost": {
    "R2": 0.5854,
    "RMSE": 8.0335,
    "MAE": 5.3464
  },
  "LightGBM": {
    "R2": 0.5302,
    "RMSE": 8.551,
    "MAE": 5.8013
  },
  "XGBoost": {
    "R2": 0.5029,
    "RMSE": 8.7966,
    "MAE": 5.8447
  },
  "best_model": "CatBoost"
}
